In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(r"..\configs")
from config_projet import C

MARGE_JOURS = 30   # meme marge que le notebook 07b de Membre A

# --- T0 et diagnostic de depart, depuis les visites de la cohorte (notebook 03) ---
visites = pd.read_csv(C.VISITES_359, dtype={"PTID": "string", "DIAGNOSIS": "string"},
                      parse_dates=["date_irm", "date_dx"])
visites["PTID"] = visites["PTID"].str.strip().str.upper()

premiere = (visites.sort_values(["PTID", "date_irm"])
            .drop_duplicates("PTID", keep="first")[["PTID", "date_irm", "date_dx", "DIAGNOSIS"]]
            .rename(columns={"date_irm": "T0", "date_dx": "date_dx_T0"})
            .reset_index(drop=True))
premiere["dx_T0"] = pd.to_numeric(premiere["DIAGNOSIS"], errors="raise").astype(int)
ptids = set(premiere["PTID"])

print("Participants :", len(premiere))
print("Ecart diagnostic <-> IRM a T0 (jours) : max",
      int((premiere["T0"] - premiere["date_dx_T0"]).dt.days.abs().max()))

In [ ]:
# --- Historique diagnostique complet (meme nettoyage que les notebooks 07 et 07b) ---
dx = pd.read_csv(C.table("All_Subjects_DXSUM_"), dtype="string", encoding="utf-8-sig")
dx["PTID"] = dx["PTID"].str.strip().str.upper()
dx["DIAGNOSIS"] = dx["DIAGNOSIS"].str.strip()
dx["date_dx"] = pd.to_datetime(dx["EXAMDATE"].str.strip(), format="%Y-%m-%d", errors="coerce")
dx = dx.loc[dx["PTID"].isin(ptids) & dx["DIAGNOSIS"].isin(["1", "2", "3"])
            & dx["date_dx"].notna(), ["PTID", "date_dx", "DIAGNOSIS"]]

conflits = dx.groupby(["PTID", "date_dx"])["DIAGNOSIS"].nunique()
conflits = conflits[conflits > 1].index
dx = dx.loc[~pd.MultiIndex.from_frame(dx[["PTID", "date_dx"]]).isin(conflits)]
dx = dx.drop_duplicates(["PTID", "date_dx"]).copy()
dx["code"] = dx["DIAGNOSIS"].astype(int)
print("Visites diagnostiques retenues :", len(dx))

# --- Visites STRICTEMENT posterieures a T0 + 30 jours ---
futur = dx.merge(premiere[["PTID", "T0"]], on="PTID", how="inner")
futur = futur.loc[futur["date_dx"] > futur["T0"] + pd.Timedelta(days=MARGE_JOURS)]

apres = futur.groupby("PTID").agg(
    dx_futur_max=("code", "max"),
    n_visites_futures=("code", "size"),
    derniere_visite=("date_dx", "max"),
)

cibles = premiere.merge(apres, on="PTID", how="left")
cibles["progression_ancree"] = np.where(
    cibles["dx_futur_max"].notna(),
    (cibles["dx_futur_max"] > cibles["dx_T0"]).astype(float), np.nan)
cibles["deja_dement"] = (cibles["dx_T0"] == 3).astype(int)
cibles["suivi_apres_T0_jours"] = (cibles["derniere_visite"] - cibles["T0"]).dt.days

print("\nDiagnostic a T0 :")
print(cibles["dx_T0"].map({1: "CN", 2: "MCI", 3: "Demence"}).value_counts().to_string())
print("\nSans visite future (cible inconnue) :", int(cibles["progression_ancree"].isna().sum()))
print("Deja dements a T0 (ne peuvent pas s'aggraver) :", int(cibles["deja_dement"].sum()))
print("\nSuivi apres T0 (annees) :")
print((cibles["suivi_apres_T0_jours"] / 365.25).describe().round(1).to_string())

In [ ]:
# --- Comparaison avec l'ancienne cible du notebook 07 ---
jeu07 = pd.read_csv(sorted(C.COHORTE.glob("preparation_modele_*"))[-1] / "jeu_modelisation_359.csv",
                    dtype={"PTID": "string"})
cmp = cibles.merge(jeu07[["PTID", "progression"]], on="PTID", how="left")

print("Ancienne cible (notebook 07) vs nouvelle (ancree) :")
display(pd.crosstab(cmp["progression"], cmp["progression_ancree"],
                    rownames=["ancienne"], colnames=["ancree"], dropna=False))

change = int((cmp["progression"] != cmp["progression_ancree"]).sum())
print("Etiquettes qui changent :", change, "/", len(cmp))

deja = cmp.loc[(cmp["progression"] == 1) & (cmp["deja_dement"] == 1)]
print("Anciens 'progresseurs' qui etaient deja dements a T0 :", len(deja))

In [ ]:
# --- Variables du tableau, mesurees AVANT T0 (jamais apres) ---
ref = cibles[["PTID", "T0"]].copy()
ref["borne"] = ref["T0"] + pd.Timedelta(days=MARGE_JOURS)

def mesure_avant_t0(prefixe, col_date, colonnes, tolerance_jours):
    """Mesure la plus recente a T0+30j ou avant, dans la tolerance donnee."""
    t = pd.read_csv(C.table(prefixe), dtype="string", encoding="utf-8-sig")
    t["PTID"] = t["PTID"].str.strip().str.upper()
    t["date"] = pd.to_datetime(t[col_date].str.strip(), errors="coerce").dt.normalize()
    noms = list(colonnes.values())
    for brut, nom in colonnes.items():
        t[nom] = pd.to_numeric(t[brut], errors="coerce")
    t = (t.loc[t["PTID"].isin(ptids)].dropna(subset=["PTID", "date"])[["PTID", "date"] + noms]
          .dropna(subset=noms, how="all").drop_duplicates(["PTID", "date"]).sort_values("date"))
    lien = pd.merge_asof(ref.sort_values("borne"), t, by="PTID",
                         left_on="borne", right_on="date",
                         direction="backward", tolerance=pd.Timedelta(days=tolerance_jours))
    return lien[["PTID"] + noms]

tab = cibles[["PTID"]].copy()
for prefixe, cd, cols, tol in [
    ("All_Subjects_MMSE_", "VISDATE", {"MMSCORE": "MMSE_debut"}, 180),
    ("All_Subjects_CDR_", "VISDATE", {"CDGLOBAL": "CDR_global", "CDRSB": "CDR_SB"}, 180),
    ("All_Subjects_ADAS_", "VISDATE", {"TOTAL13": "ADAS13"}, 180),
    ("All_Subjects_FAQ_", "VISDATE", {"FAQTOTAL": "FAQ_debut"}, 180),
    ("All_Subjects_UPENNBIOMK_ROCHE_", "EXAMDATE",
     {"ABETA42": "ABETA42", "TAU": "TAU", "PTAU": "PTAU"}, 365),
]:
    tab = tab.merge(mesure_avant_t0(prefixe, cd, cols, tol), on="PTID",
                    how="left", validate="one_to_one")

# Demographie + age a T0.
def premier(s):
    s = s.str.strip().replace("", pd.NA).dropna()
    return s.iloc[0] if len(s) else pd.NA

demo = pd.read_csv(C.table("All_Subjects_PTDEMOG_"), dtype="string", encoding="utf-8-sig")
demo["PTID"] = demo["PTID"].str.strip().str.upper()
d = (demo.loc[demo["PTID"].isin(ptids)].groupby("PTID")
     .agg(sexe=("PTGENDER", premier), education=("PTEDUCAT", premier),
          naissance=("PTDOBYY", premier)).reset_index())
d["annee_naissance"] = pd.to_numeric(
    d["naissance"].astype("string").str.extract(r"(\d{4})")[0], errors="coerce")
d["sexe"] = pd.to_numeric(d["sexe"], errors="coerce")
d["education"] = pd.to_numeric(d["education"], errors="coerce")
tab = tab.merge(d.drop(columns="naissance"), on="PTID", how="left", validate="one_to_one")
tab = tab.merge(cibles[["PTID", "T0"]], on="PTID")
tab["age_reference"] = tab["T0"].dt.year - tab["annee_naissance"]

# APOE : nombre d'alleles e4.
apoe = pd.read_csv(C.table("All_Subjects_APOERES_"), dtype="string", encoding="utf-8-sig")
apoe["PTID"] = apoe["PTID"].str.strip().str.upper()
apoe = (apoe.loc[apoe["PTID"].isin(ptids)].groupby("PTID")["GENOTYPE"]
        .agg(premier).rename("APOE_genotype").reset_index())
apoe["APOE_e4"] = apoe["APOE_genotype"].map(
    lambda g: pd.NA if pd.isna(g) or "/" not in str(g)
    else sum(a.strip() == "4" for a in str(g).split("/"))).astype("Int64")
tab = tab.merge(apoe, on="PTID", how="left", validate="one_to_one").drop(columns="T0")

remplissage = pd.DataFrame({"valeurs": tab.notna().sum(),
                            "taux_%": (tab.notna().mean() * 100).round(1)})
display(remplissage.sort_values("taux_%", ascending=False))
print("Age min / median / max :",
      tab["age_reference"].min(), tab["age_reference"].median(), tab["age_reference"].max())

In [ ]:
# --- Assemblage et enregistrement ---
jeu = (cibles.merge(tab, on="PTID", validate="one_to_one")
             .merge(jeu07[["PTID", "progression"]].rename(
                 columns={"progression": "progression_ancienne"}),
                 on="PTID", how="left", validate="one_to_one"))

SORTIE = C.sortie("cible_ancree_359")
jeu.to_csv(SORTIE / "jeu_modelisation_359_ancre.csv", index=False, encoding="utf-8-sig")

a_risque = jeu.loc[(jeu["deja_dement"] == 0) & jeu["progression_ancree"].notna()]
print("Jeu :", jeu.shape)
print("\nPopulation 'a risque' (pas encore demente a T0, cible connue) :", len(a_risque))
print("   dont progressent :", int(a_risque["progression_ancree"].sum()),
      f"({a_risque['progression_ancree'].mean() * 100:.1f}%)")
print("\nFichier :", SORTIE / "jeu_modelisation_359_ancre.csv")
display(jeu.head())